# Q3 carrier and proportional-hazards sensitivity

**Question:** How do the retained carrier models and PH sensitivity analyses support interpretation of the CVD results?

**Inputs:** Q3 model-preparation CVD/CRP datasets.

**Outputs:** Carrier estimates, PH diagnostics and the specified lighter PH sensitivity models.

Use a fresh R kernel. These are carrier-coded analyses, distinct from additive ALT dosage. Original formulas and model-family output directories are preserved. Retained source contains plotting next to diagnostics; those plots remain with the model checks rather than being refitted elsewhere.


## Carrier analysis and PH diagnostics


In [ ]:
TIER1_SNPS <- c("rs1544410", "rs2228570", "rs731236", "rs7975232")
REBUILD_OUTPUT_DIR_NAME <- "Q3_VDR_v9_rebuild_outputs"
MODEL_PREP_OUTPUT_DIR_NAME <- "Q3_VDR_v9_model_prep_outputs"
TIER1_OUTPUT_DIR_NAME <- "Q3_VDR_tier1_model_outputs_R"

OVERWRITE_OUTPUTS <- tolower(Sys.getenv("Q3_VDR_OVERWRITE_OUTPUTS", "true")) %in% c("1", "true", "yes", "y")

infer_default_data_dir <- function() {
  cwd <- getwd()
  candidates <- c(
    cwd,
    file.path(cwd, "data"),
    "/home/jupyter/workspace/VDR",
    "/home/jupyter/workspace/VDR/data",
    "/home/jupyter/workspace/Q3_VDR",
    "/home/jupyter/workspace/Q3_VDR/data"
  )
  for (p in candidates) {
    if (dir.exists(p)) {
      return(p)
    }
  }
  cwd
}

DATA_DIR <- Sys.getenv("Q3_VDR_DATA_DIR", infer_default_data_dir())
MODEL_PREP_DIR <- Sys.getenv("Q3_VDR_MODEL_OUTPUT_DIR", file.path(DATA_DIR, MODEL_PREP_OUTPUT_DIR_NAME))
REBUILD_DIR <- Sys.getenv("Q3_VDR_REBUILD_OUTPUT_DIR", file.path(DATA_DIR, REBUILD_OUTPUT_DIR_NAME))
TIER1_DIR <- Sys.getenv("Q3_VDR_TIER1_R_MODEL_OUTPUT_DIR", file.path(DATA_DIR, TIER1_OUTPUT_DIR_NAME))
dir.create(TIER1_DIR, recursive = TRUE, showWarnings = FALSE)

save_csv <- function(df, filename) {
  path <- file.path(TIER1_DIR, filename)
  existed <- file.exists(path)
  if (existed && !OVERWRITE_OUTPUTS) {
    stop(sprintf("Output exists and overwrite is disabled: %s", path))
  }
  write.csv(df, path, row.names = FALSE, na = "")
  action <- if (existed) "overwritten" else "created"
  message(sprintf("%s: %s rows=%s cols=%s", action, path, nrow(df), ncol(df)))
  path
}

pkg_status <- data.frame(
  package = c("survival", "stats"),
  available = c(requireNamespace("survival", quietly = TRUE), TRUE),
  stringsAsFactors = FALSE
)
if (!pkg_status$available[pkg_status$package == "survival"]) {
  stop("R package 'survival' is required for coxph but is not available.")
}
library(survival)

run_config <- data.frame(
  setting = c(
    "DATA_DIR", "MODEL_PREP_DIR", "REBUILD_DIR", "TIER1_DIR",
    "OVERWRITE_OUTPUTS", "TIER1_SNPS", "model_engine", "scope_freeze", "output_policy"
  ),
  value = c(
    DATA_DIR, MODEL_PREP_DIR, REBUILD_DIR, TIER1_DIR,
    as.character(OVERWRITE_OUTPUTS), paste(TIER1_SNPS, collapse = ";"),
    "R survival::coxph + stats::lm HC3",
    "Tier 1 VDR SNP x measured 25OHD only; no Tier 2, Xiao/Wang, GRS, or MR.",
    "overwrite same-name CSV files in TIER1_DIR; do not delete unrelated files"
  ),
  stringsAsFactors = FALSE
)
save_csv(run_config, "00_tier1_R_model_run_configuration.csv")
save_csv(pkg_status, "00_tier1_R_package_status.csv")

cat("Reading model-prep inputs from:", MODEL_PREP_DIR, "\n")
cat("Writing Tier 1 R model outputs to:", TIER1_DIR, "\n")

read_csv_required <- function(path, label) {
  if (!file.exists(path)) {
    stop(sprintf("Missing required input for %s: %s", label, path))
  }
  df <- read.csv(path, check.names = FALSE, stringsAsFactors = FALSE)
  message(sprintf("Loaded %s: rows=%s cols=%s from %s", label, nrow(df), ncol(df), path))
  df
}

read_csv_optional <- function(path, label) {
  if (!file.exists(path)) {
    message(sprintf("Missing optional input for %s: %s", label, path))
    return(data.frame())
  }
  df <- read.csv(path, check.names = FALSE, stringsAsFactors = FALSE)
  message(sprintf("Loaded %s: rows=%s cols=%s from %s", label, nrow(df), ncol(df), path))
  df
}

cvd <- read_csv_required(file.path(MODEL_PREP_DIR, "02_cvd_model_dataset_v9.csv"), "CVD model-prep dataset")
crp <- read_csv_optional(file.path(MODEL_PREP_DIR, "03_crp_model_dataset_v9.csv"), "CRP model-prep dataset")
candidate_plan <- read_csv_optional(file.path(MODEL_PREP_DIR, "01_candidate_analysis_plan_v9.csv"), "candidate analysis plan")
prep_summary <- read_csv_optional(file.path(MODEL_PREP_DIR, "04_model_dataset_summary_v9.csv"), "model dataset summary")

tier1_carrier_cols <- paste0(TIER1_SNPS, "_carrier")
missing_tier1 <- setdiff(tier1_carrier_cols, names(cvd))
if (length(missing_tier1) > 0) {
  stop(sprintf("Missing Tier 1 carrier columns in CVD model dataset: %s", paste(missing_tier1, collapse = ";")))
}

present_carrier_cols <- names(cvd)[grepl("^rs.*_carrier$", names(cvd))]
non_tier1_carrier_cols_present <- setdiff(present_carrier_cols, tier1_carrier_cols)
scope_status <- data.frame(
  item = c(
    "tier1_snps",
    "tier1_carrier_columns_present",
    "non_tier1_carrier_columns_present_but_not_used",
    "tier2_xiao_grs_mr_scope"
  ),
  value = c(
    paste(TIER1_SNPS, collapse = ";"),
    paste(tier1_carrier_cols, collapse = ";"),
    paste(non_tier1_carrier_cols_present, collapse = ";"),
    "not used in this notebook"
  ),
  status = c("included", "yes", "deferred", "excluded"),
  stringsAsFactors = FALSE
)
save_csv(scope_status, "01_tier1_R_scope_status.csv")

numeric_cols <- c(
  "vitd_main", "log_vitd", "vitd_lt20", "vitd_per10", "age_at_index", "female", "winter", "bmi_full",
  "follow_up_years", "event_mi_or_stroke", "baseline_diabetes", "baseline_hypertension",
  "baseline_hyperlipidemia", "crp_mgL", "log_crp", tier1_carrier_cols
)

coerce_numeric <- function(df, cols) {
  for (col in intersect(cols, names(df))) {
    df[[col]] <- suppressWarnings(as.numeric(df[[col]]))
  }
  df
}

cvd <- coerce_numeric(cvd, numeric_cols)
if (nrow(crp) > 0) {
  crp <- coerce_numeric(crp, numeric_cols)
}

for (rsid in TIER1_SNPS) {
  carrier_col <- paste0(rsid, "_carrier")
  ix_bin <- paste0(rsid, "_x_vitd_lt20")
  ix_log <- paste0(rsid, "_x_log_vitd")
  # Preserve NA carrier values if any; unknown genotype calls are not non-carriers.
  cvd[[carrier_col]] <- as.integer(cvd[[carrier_col]])
  cvd[[ix_bin]] <- cvd$vitd_lt20 * cvd[[carrier_col]]
  cvd[[ix_log]] <- cvd$log_vitd * cvd[[carrier_col]]
  if (nrow(crp) > 0 && carrier_col %in% names(crp)) {
    # Preserve NA carrier values if any; complete-case filtering will exclude them.
    crp[[carrier_col]] <- as.integer(crp[[carrier_col]])
    crp[[ix_bin]] <- crp$vitd_lt20 * crp[[carrier_col]]
    crp[[ix_log]] <- crp$log_vitd * crp[[carrier_col]]
  }
}

dataset_summary <- data.frame(
  dataset = c("cvd", "crp"),
  rows = c(nrow(cvd), nrow(crp)),
  events_mi_or_stroke = c(
    if ("event_mi_or_stroke" %in% names(cvd)) sum(cvd$event_mi_or_stroke, na.rm = TRUE) else NA,
    if (nrow(crp) > 0 && "event_mi_or_stroke" %in% names(crp)) sum(crp$event_mi_or_stroke, na.rm = TRUE) else NA
  ),
  columns = c(ncol(cvd), ncol(crp)),
  stringsAsFactors = FALSE
)
save_csv(dataset_summary, "02_tier1_R_input_dataset_summary.csv")
scope_status

holm_adjust <- function(p) {
  out <- rep(NA_real_, length(p))
  ok <- which(!is.na(p))
  if (length(ok) > 0) {
    out[ok] <- p.adjust(p[ok], method = "holm")
  }
  out
}

bind_rows_base <- function(rows) {
  rows <- rows[vapply(rows, function(x) is.data.frame(x) && nrow(x) > 0, logical(1))]
  if (length(rows) == 0) {
    return(data.frame())
  }
  all_cols <- unique(unlist(lapply(rows, names)))
  aligned <- lapply(rows, function(d) {
    missing_cols <- setdiff(all_cols, names(d))
    for (col in missing_cols) {
      d[[col]] <- NA
    }
    d[, all_cols, drop = FALSE]
  })
  do.call(rbind, aligned)
}

build_formula <- function(outcome, numeric_terms, categorical_terms = character(), survival = FALSE) {
  numeric_terms <- numeric_terms[!is.na(numeric_terms) & nzchar(numeric_terms)]
  categorical_terms <- categorical_terms[!is.na(categorical_terms) & nzchar(categorical_terms)]
  categorical_rhs <- if (length(categorical_terms) > 0) paste0("factor(", categorical_terms, ")") else character(0)
  rhs <- c(numeric_terms, categorical_rhs)
  if (survival) {
    as.formula(paste0("Surv(follow_up_years, event_mi_or_stroke) ~ ", paste(rhs, collapse = " + ")))
  } else {
    as.formula(paste0(outcome, " ~ ", paste(rhs, collapse = " + ")))
  }
}

complete_case_data <- function(df, needed) {
  needed <- unique(needed[!is.na(needed) & nzchar(needed)])
  missing_terms <- setdiff(needed, names(df))
  if (length(missing_terms) > 0) {
    return(list(data = data.frame(), status = "missing_terms", detail = paste(missing_terms, collapse = ";")))
  }
  d <- df[stats::complete.cases(df[, needed, drop = FALSE]), , drop = FALSE]
  if (nrow(d) == 0) {
    return(list(data = d, status = "no_complete_cases", detail = ""))
  }
  list(data = d, status = "ready", detail = "")
}

cox_result_table <- function(fit) {
  s <- summary(fit)
  coefs <- as.data.frame(s$coefficients)
  coefs$term <- rownames(coefs)
  rownames(coefs) <- NULL
  out <- data.frame(
    term = coefs$term,
    coef = coefs$coef,
    se = coefs$`se(coef)`,
    z_or_t = coefs$z,
    p = coefs$`Pr(>|z|)`,
    stringsAsFactors = FALSE
  )
  out$ci_lower_coef <- out$coef - 1.96 * out$se
  out$ci_upper_coef <- out$coef + 1.96 * out$se
  out$hr <- exp(out$coef)
  out$hr_ci_lower <- exp(out$ci_lower_coef)
  out$hr_ci_upper <- exp(out$ci_upper_coef)
  out
}

vcov_hc3 <- function(fit) {
  X <- model.matrix(fit)
  e <- residuals(fit)
  h <- hatvalues(fit)
  w <- as.numeric((e / pmax(1 - h, .Machine$double.eps))^2)
  xtx <- crossprod(X)
  bread <- tryCatch(solve(xtx), error = function(e) qr.solve(xtx))
  meat <- crossprod(X, X * w)
  bread %*% meat %*% bread
}

lm_hc3_result_table <- function(fit) {
  beta <- coef(fit)
  vc <- vcov_hc3(fit)
  se <- sqrt(diag(vc))
  tval <- beta / se
  pval <- 2 * pt(abs(tval), df = df.residual(fit), lower.tail = FALSE)
  out <- data.frame(
    term = names(beta),
    coef = as.numeric(beta),
    se = as.numeric(se),
    z_or_t = as.numeric(tval),
    p = as.numeric(pval),
    stringsAsFactors = FALSE
  )
  out <- out[out$term != "(Intercept)", , drop = FALSE]
  out$ci_lower_coef <- out$coef - 1.96 * out$se
  out$ci_upper_coef <- out$coef + 1.96 * out$se
  out$exp_coef <- exp(out$coef)
  out$percent_change <- (exp(out$coef) - 1) * 100
  list(table = out, vcov = vc)
}

fit_cox_model <- function(df, spec) {
  needed <- unique(c("follow_up_years", "event_mi_or_stroke", spec$numeric_terms, spec$categorical_terms))
  cc <- complete_case_data(df, needed)
  info <- list(status = cc$status, detail = cc$detail, n = nrow(cc$data), events = NA, n_predictors = NA, backend = "survival::coxph")
  if (cc$status != "ready") {
    return(list(result = data.frame(), vcov = matrix(numeric(0), 0, 0), fit = NULL, info = info))
  }
  d <- cc$data
  info$events <- sum(d$event_mi_or_stroke, na.rm = TRUE)
  if (info$events < 20) {
    info$status <- "too_few_events"
    return(list(result = data.frame(), vcov = matrix(numeric(0), 0, 0), fit = NULL, info = info))
  }
  f <- build_formula(NULL, spec$numeric_terms, spec$categorical_terms, survival = TRUE)
  fit <- tryCatch(coxph(f, data = d, ties = "efron", x = TRUE, model = TRUE), error = function(e) e)
  if (inherits(fit, "error")) {
    info$status <- "fit_error"
    info$detail <- conditionMessage(fit)
    return(list(result = data.frame(), vcov = matrix(numeric(0), 0, 0), fit = NULL, info = info))
  }
  result <- cox_result_table(fit)
  info$status <- "pass"
  info$n_predictors <- length(coef(fit))
  list(result = result, vcov = vcov(fit), fit = fit, info = info)
}

fit_lm_model <- function(df, spec) {
  needed <- unique(c(spec$outcome, spec$numeric_terms, spec$categorical_terms))
  cc <- complete_case_data(df, needed)
  info <- list(status = cc$status, detail = cc$detail, n = nrow(cc$data), events = NA, n_predictors = NA, backend = "stats::lm_hc3")
  if (cc$status != "ready") {
    return(list(result = data.frame(), vcov = matrix(numeric(0), 0, 0), fit = NULL, info = info))
  }
  d <- cc$data
  f <- build_formula(spec$outcome, spec$numeric_terms, spec$categorical_terms, survival = FALSE)
  fit <- tryCatch(lm(f, data = d), error = function(e) e)
  if (inherits(fit, "error")) {
    info$status <- "fit_error"
    info$detail <- conditionMessage(fit)
    return(list(result = data.frame(), vcov = matrix(numeric(0), 0, 0), fit = NULL, info = info))
  }
  hc3 <- lm_hc3_result_table(fit)
  info$status <- "pass"
  info$n_predictors <- length(coef(fit)) - 1
  info$r_squared <- summary(fit)$r.squared
  list(result = hc3$table, vcov = hc3$vcov, fit = fit, info = info)
}

linear_combo <- function(label, weights, coef_vec, vc) {
  terms <- names(weights)
  if (length(terms) == 0 || any(!(terms %in% names(coef_vec))) || any(!(terms %in% rownames(vc)))) {
    return(data.frame(contrast = label, status = "missing_terms", stringsAsFactors = FALSE))
  }
  w <- as.numeric(weights)
  names(w) <- terms
  beta <- sum(w * coef_vec[terms])
  sub_v <- vc[terms, terms, drop = FALSE]
  variance <- as.numeric(t(w) %*% sub_v %*% w)
  se <- ifelse(variance >= 0, sqrt(variance), NA_real_)
  data.frame(
    contrast = label,
    coef = beta,
    se = se,
    ci_lower_coef = beta - 1.96 * se,
    ci_upper_coef = beta + 1.96 * se,
    hr_or_exp_coef = exp(beta),
    ci_lower = exp(beta - 1.96 * se),
    ci_upper = exp(beta + 1.96 * se),
    status = "pass",
    stringsAsFactors = FALSE
  )
}

specs <- list()
idx <- 1
add_spec <- function(analysis_family, analysis_role, model_name, rsid, exposure, outcome = NA, numeric_terms, categorical_terms = character()) {
  carrier <- paste0(rsid, "_carrier")
  interaction <- if (exposure == "vitd_lt20") paste0(rsid, "_x_vitd_lt20") else paste0(rsid, "_x_log_vitd")
  specs[[idx <<- idx]] <<- list(
    analysis_family = analysis_family,
    analysis_role = analysis_role,
    model_name = model_name,
    rsid = rsid,
    exposure = exposure,
    outcome = outcome,
    carrier = carrier,
    interaction = interaction,
    numeric_terms = numeric_terms,
    categorical_terms = categorical_terms
  )
  idx <<- idx + 1
}

for (rsid in TIER1_SNPS) {
  carrier <- paste0(rsid, "_carrier")
  ix_bin <- paste0(rsid, "_x_vitd_lt20")
  ix_log <- paste0(rsid, "_x_log_vitd")
  add_spec("cvd_cox", "primary", "cvd_binary_core_with_bmi", rsid, "vitd_lt20",
           numeric_terms = c("vitd_lt20", carrier, ix_bin, "age_at_index", "female", "winter", "bmi_full"))
  add_spec("cvd_cox", "missingness_sensitivity", "cvd_binary_core_no_bmi", rsid, "vitd_lt20",
           numeric_terms = c("vitd_lt20", carrier, ix_bin, "age_at_index", "female", "winter"))
  add_spec("cvd_cox", "continuous_vitd_sensitivity", "cvd_logvitd_core_with_bmi", rsid, "log_vitd",
           numeric_terms = c("log_vitd", carrier, ix_log, "age_at_index", "female", "winter", "bmi_full"))
  add_spec("cvd_cox", "extended_covariate_sensitivity", "cvd_binary_extended_with_bmi_race_clinical", rsid, "vitd_lt20",
           numeric_terms = c("vitd_lt20", carrier, ix_bin, "age_at_index", "female", "winter", "bmi_full", "baseline_diabetes", "baseline_hypertension", "baseline_hyperlipidemia"),
           categorical_terms = c("race_eth_group"))
  add_spec("crp_ols", "crp_extension", "crp_loglog_core_with_bmi", rsid, "log_vitd", outcome = "log_crp",
           numeric_terms = c("log_vitd", carrier, ix_log, "age_at_index", "female", "winter", "bmi_full"))
}

spec_rows <- do.call(rbind, lapply(specs, function(s) {
  data.frame(
    analysis_family = s$analysis_family,
    analysis_role = s$analysis_role,
    model_name = s$model_name,
    rsid = s$rsid,
    exposure = s$exposure,
    outcome = ifelse(is.na(s$outcome), "", s$outcome),
    carrier = s$carrier,
    interaction = s$interaction,
    numeric_terms = paste(s$numeric_terms, collapse = ";"),
    categorical_terms = paste(s$categorical_terms, collapse = ";"),
    scope = "Tier 1 only",
    stringsAsFactors = FALSE
  )
}))
save_csv(spec_rows, "03_tier1_R_model_specifications.csv")
head(spec_rows, 30)

all_results <- list()
all_status <- list()
joint_rows <- list()
ph_rows <- list()
figure_rows <- list()
r_i <- 1
s_i <- 1
j_i <- 1
ph_i <- 1
fig_i <- 1

save_zph_plot <- function(zph, spec) {
  filename <- paste0("24_tier1_R_cox_zph_", spec$rsid, "_", spec$model_name, ".pdf")
  path <- file.path(TIER1_DIR, filename)
  status <- "created"
  detail <- ""
  tryCatch({
    grDevices::pdf(path, width = 10, height = 8)
    on.exit(grDevices::dev.off(), add = TRUE)
    plot(zph)
  }, error = function(e) {
    status <<- "plot_error"
    detail <<- conditionMessage(e)
  })
  data.frame(
    output_file = filename,
    path = path,
    figure_type = "cox_zph_residual_diagnostics",
    rsid = spec$rsid,
    model_name = spec$model_name,
    status = status,
    detail = detail,
    n_rows = NA,
    stringsAsFactors = FALSE
  )
}

for (spec in specs) {
  if (spec$analysis_family == "cvd_cox") {
    fit_obj <- fit_cox_model(cvd, spec)
  } else if (spec$analysis_family == "crp_ols") {
    if (nrow(crp) == 0) {
      fit_obj <- list(result = data.frame(), vcov = matrix(numeric(0), 0, 0), fit = NULL,
                      info = list(status = "crp_dataset_missing", detail = "", n = 0, events = NA, n_predictors = NA, backend = "stats::lm_hc3"))
    } else {
      fit_obj <- fit_lm_model(crp, spec)
    }
  } else {
    fit_obj <- list(result = data.frame(), vcov = matrix(numeric(0), 0, 0), fit = NULL,
                    info = list(status = "unknown_analysis_family", detail = spec$analysis_family, n = NA, events = NA, n_predictors = NA, backend = ""))
  }

  info <- fit_obj$info
  all_status[[s_i]] <- data.frame(
    analysis_family = spec$analysis_family,
    analysis_role = spec$analysis_role,
    model_name = spec$model_name,
    rsid = spec$rsid,
    status = info$status,
    detail = info$detail,
    n = info$n,
    events = info$events,
    n_predictors = info$n_predictors,
    backend = info$backend,
    r_squared = ifelse(is.null(info$r_squared), NA, info$r_squared),
    stringsAsFactors = FALSE
  )
  s_i <- s_i + 1

  if (nrow(fit_obj$result) > 0) {
    res <- fit_obj$result
    res$analysis_family <- spec$analysis_family
    res$analysis_role <- spec$analysis_role
    res$model_name <- spec$model_name
    res$rsid <- spec$rsid
    res$n <- info$n
    res$events <- info$events
    res$backend <- info$backend
    res$exposure <- spec$exposure
    res$carrier <- spec$carrier
    res$interaction <- spec$interaction
    all_results[[r_i]] <- res
    r_i <- r_i + 1

    coef_vec <- coef(fit_obj$fit)
    vc <- fit_obj$vcov
    if (spec$analysis_family == "cvd_cox" && spec$exposure == "vitd_lt20") {
      contrasts <- list(
        linear_combo("vitamin_D_deficiency_HR_among_noncarriers", c(vitd_lt20 = 1), coef_vec, vc),
        linear_combo("vitamin_D_deficiency_HR_among_carriers", setNames(c(1, 1), c("vitd_lt20", spec$interaction)), coef_vec, vc),
        linear_combo("carrier_HR_among_nondeficient", setNames(1, spec$carrier), coef_vec, vc),
        linear_combo("carrier_HR_among_deficient", setNames(c(1, 1), c(spec$carrier, spec$interaction)), coef_vec, vc),
        linear_combo("multiplicative_interaction_ratio", setNames(1, spec$interaction), coef_vec, vc)
      )
      for (ct in contrasts) {
        ct$analysis_family <- spec$analysis_family
        ct$analysis_role <- spec$analysis_role
        ct$model_name <- spec$model_name
        ct$rsid <- spec$rsid
        ct$n <- info$n
        ct$events <- info$events
        joint_rows[[j_i]] <- ct
        j_i <- j_i + 1
      }
    } else if (spec$analysis_family == "crp_ols") {
      contrasts <- list(
        linear_combo("log_vitd_slope_among_noncarriers", c(log_vitd = 1), coef_vec, vc),
        linear_combo("log_vitd_slope_among_carriers", setNames(c(1, 1), c("log_vitd", spec$interaction)), coef_vec, vc),
        linear_combo("carrier_difference_at_log_vitd_zero", setNames(1, spec$carrier), coef_vec, vc),
        linear_combo("log_vitd_x_carrier_interaction", setNames(1, spec$interaction), coef_vec, vc)
      )
      for (ct in contrasts) {
        ct$analysis_family <- spec$analysis_family
        ct$analysis_role <- spec$analysis_role
        ct$model_name <- spec$model_name
        ct$rsid <- spec$rsid
        ct$n <- info$n
        ct$events <- NA
        joint_rows[[j_i]] <- ct
        j_i <- j_i + 1
      }
    }

    if (spec$analysis_family == "cvd_cox" && !is.null(fit_obj$fit)) {
      zph <- tryCatch(cox.zph(fit_obj$fit), error = function(e) e)
      if (!inherits(zph, "error")) {
        zph_table <- as.data.frame(zph$table)
        zph_table$term <- rownames(zph_table)
        rownames(zph_table) <- NULL
        names(zph_table) <- make.names(names(zph_table))
        zph_table$analysis_family <- spec$analysis_family
        zph_table$analysis_role <- spec$analysis_role
        zph_table$model_name <- spec$model_name
        zph_table$rsid <- spec$rsid
        ph_rows[[ph_i]] <- zph_table
        ph_i <- ph_i + 1
        if (spec$analysis_role == "primary") {
          figure_rows[[fig_i]] <- save_zph_plot(zph, spec)
          fig_i <- fig_i + 1
        }
      } else {
        ph_rows[[ph_i]] <- data.frame(
          analysis_family = spec$analysis_family,
          analysis_role = spec$analysis_role,
          model_name = spec$model_name,
          rsid = spec$rsid,
          term = "cox.zph_error",
          detail = conditionMessage(zph),
          stringsAsFactors = FALSE
        )
        ph_i <- ph_i + 1
      }
    }
  }
}

results_df <- bind_rows_base(all_results)
status_df <- bind_rows_base(all_status)
joint_df <- bind_rows_base(joint_rows)
ph_df <- bind_rows_base(ph_rows)
figure_df <- bind_rows_base(figure_rows)

save_csv(status_df, "04_tier1_R_model_fit_status.csv")
save_csv(results_df, "05_tier1_R_all_model_term_results.csv")
save_csv(joint_df, "06_tier1_R_joint_and_stratum_contrasts.csv")
save_csv(ph_df, "14_tier1_R_cox_ph_assumption_zph.csv")

primary <- if (nrow(results_df) > 0) {
  subset(results_df, analysis_family == "cvd_cox" & analysis_role == "primary")
} else data.frame()
sensitivity <- if (nrow(results_df) > 0) {
  subset(results_df, analysis_family == "cvd_cox" & analysis_role != "primary")
} else data.frame()
crp_results <- if (nrow(results_df) > 0) {
  subset(results_df, analysis_family == "crp_ols")
} else data.frame()

save_csv(primary, "07_tier1_R_cvd_cox_primary_term_results.csv")
save_csv(sensitivity, "08_tier1_R_cvd_cox_sensitivity_term_results.csv")
save_csv(crp_results, "09_tier1_R_crp_lm_term_results.csv")

head(status_df, 30)

if (nrow(primary) > 0) {
  primary_interactions <- primary[primary$term == primary$interaction, , drop = FALSE]
  primary_interactions$p_bonferroni_4 <- pmin(as.numeric(primary_interactions$p) * length(TIER1_SNPS), 1)
  primary_interactions$p_holm_4 <- holm_adjust(as.numeric(primary_interactions$p))
  primary_interactions$multiple_testing_family <- "four Tier 1 VDR primary interaction tests"
} else {
  primary_interactions <- data.frame()
}

primary_joint <- if (nrow(joint_df) > 0) {
  subset(joint_df, analysis_family == "cvd_cox" & analysis_role == "primary")
} else data.frame()

make_figure_row <- function(filename, figure_type, status, detail = "", n_rows = NA) {
  data.frame(
    output_file = filename,
    path = file.path(TIER1_DIR, filename),
    figure_type = figure_type,
    rsid = "",
    model_name = "",
    status = status,
    detail = detail,
    n_rows = n_rows,
    stringsAsFactors = FALSE
  )
}

plot_forest <- function(df, filename, figure_type, label_col, estimate_col, lower_col, upper_col, title, xlab) {
  needed <- c(label_col, estimate_col, lower_col, upper_col)
  if (nrow(df) == 0 || any(!(needed %in% names(df)))) {
    return(make_figure_row(filename, figure_type, "no_data", "required columns missing or empty input", 0))
  }
  est <- suppressWarnings(as.numeric(df[[estimate_col]]))
  lo <- suppressWarnings(as.numeric(df[[lower_col]]))
  hi <- suppressWarnings(as.numeric(df[[upper_col]]))
  ok <- is.finite(est) & is.finite(lo) & is.finite(hi) & lo > 0 & hi > 0
  d <- df[ok, , drop = FALSE]
  est <- est[ok]
  lo <- lo[ok]
  hi <- hi[ok]
  if (nrow(d) == 0) {
    return(make_figure_row(filename, figure_type, "no_plottable_rows", "no finite positive HR/CI values", 0))
  }
  labels <- as.character(d[[label_col]])
  ord <- order(est)
  est <- est[ord]
  lo <- lo[ord]
  hi <- hi[ord]
  labels <- labels[ord]
  path <- file.path(TIER1_DIR, filename)
  status <- "created"
  detail <- ""
  tryCatch({
    grDevices::png(path, width = 2100, height = max(900, 260 + 95 * length(labels)), res = 160)
    on.exit(grDevices::dev.off(), add = TRUE)
    par(mar = c(5, 11, 4, 2))
    y <- seq_along(labels)
    log_range <- range(log(c(lo, hi, 1)), finite = TRUE)
    pad <- max(0.12, diff(log_range) * 0.08)
    xlim <- exp(log_range + c(-pad, pad))
    plot(est, y, log = "x", xlim = xlim, yaxt = "n", pch = 19,
         xlab = xlab, ylab = "", main = title)
    segments(lo, y, hi, y, lwd = 1.6)
    abline(v = 1, lty = 2, col = "gray45")
    axis(2, at = y, labels = labels, las = 1, cex.axis = 0.72)
  }, error = function(e) {
    status <<- "plot_error"
    detail <<- conditionMessage(e)
  })
  make_figure_row(filename, figure_type, status, detail, length(labels))
}

plot_ph_global <- function(ph_df) {
  filename <- "23_tier1_R_cox_ph_global_pvalues.png"
  if (nrow(ph_df) == 0 || !("p" %in% names(ph_df)) || !("term" %in% names(ph_df))) {
    return(make_figure_row(filename, "cox_ph_global_pvalue_plot", "no_data", "cox.zph table missing", 0))
  }
  d <- subset(ph_df, analysis_role == "primary" & term == "GLOBAL")
  d$p <- suppressWarnings(as.numeric(d$p))
  d <- d[is.finite(d$p), , drop = FALSE]
  if (nrow(d) == 0) {
    return(make_figure_row(filename, "cox_ph_global_pvalue_plot", "no_plottable_rows", "no finite GLOBAL p-values", 0))
  }
  path <- file.path(TIER1_DIR, filename)
  status <- "created"
  detail <- ""
  tryCatch({
    grDevices::png(path, width = 1500, height = 950, res = 160)
    on.exit(grDevices::dev.off(), add = TRUE)
    vals <- -log10(pmax(d$p, .Machine$double.xmin))
    par(mar = c(7, 5, 4, 2))
    barplot(vals, names.arg = d$rsid, las = 2, ylab = "-log10(P)",
            main = "Cox PH global tests: Tier 1 primary models")
    abline(h = -log10(0.05), lty = 2, col = "gray45")
  }, error = function(e) {
    status <<- "plot_error"
    detail <<- conditionMessage(e)
  })
  make_figure_row(filename, "cox_ph_global_pvalue_plot", status, detail, nrow(d))
}

figure_extra <- list()
figure_extra[[1]] <- plot_forest(
  primary_interactions,
  "21_tier1_R_primary_interaction_forest.png",
  "primary_interaction_hr_forest",
  "rsid", "hr", "hr_ci_lower", "hr_ci_upper",
  "VDR x vitamin D deficiency interaction HR ratios",
  "Multiplicative interaction HR ratio, log scale"
)

joint_plot <- primary_joint
if (nrow(joint_plot) > 0 && "contrast" %in% names(joint_plot)) {
  keep_contrasts <- c(
    "vitamin_D_deficiency_HR_among_noncarriers",
    "vitamin_D_deficiency_HR_among_carriers",
    "multiplicative_interaction_ratio"
  )
  joint_plot <- joint_plot[joint_plot$contrast %in% keep_contrasts, , drop = FALSE]
  contrast_label <- c(
    vitamin_D_deficiency_HR_among_noncarriers = "VitD<20 HR noncarrier",
    vitamin_D_deficiency_HR_among_carriers = "VitD<20 HR carrier",
    multiplicative_interaction_ratio = "interaction ratio"
  )
  joint_plot$short_contrast <- contrast_label[joint_plot$contrast]
  joint_plot$plot_label <- paste(joint_plot$rsid, joint_plot$short_contrast, sep = " | ")
}
figure_extra[[2]] <- plot_forest(
  joint_plot,
  "22_tier1_R_primary_joint_hr_forest.png",
  "primary_joint_hr_forest",
  "plot_label", "hr_or_exp_coef", "ci_lower", "ci_upper",
  "Tier 1 primary joint HR contrasts",
  "HR or interaction ratio, log scale"
)
figure_extra[[3]] <- plot_ph_global(ph_df)

figure_extra_df <- bind_rows_base(figure_extra)
figure_df <- bind_rows_base(list(figure_df, figure_extra_df))

complete_case_counts <- status_df[, c("analysis_family", "analysis_role", "model_name", "rsid", "n", "events", "status")]
names(complete_case_counts)[names(complete_case_counts) == "n"] <- "complete_case_n"

interpretation_flags <- data.frame(
  item = c("scope", "genetic_coding", "primary_estimand", "population_stratification", "multiple_testing", "bmi_missingness", "cox_diagnostics"),
  status = c("Tier 1 only", "dominant_ALT_carrier", "effect_modification", "not_fully_solved", "Tier 1 family only", "important", "cox.zph_output_created"),
  note = c(
    "Expanded VDR, Xiao/Wang pathway, weighted GRS, genotype extraction, and MR are deferred.",
    "Carrier status is not additive genotype dosage.",
    "Interaction is interpreted as effect modification in observational data, not causal proof.",
    "No genetic ancestry PCs are included; race/ethnicity is only used in extended sensitivity.",
    "Primary interaction p-values adjusted across four Tier 1 SNPs.",
    "BMI-adjusted primary model is complete-case; no-BMI model is missingness sensitivity.",
    "Schoenfeld residual tests are saved for Cox models."
  ),
  stringsAsFactors = FALSE
)

save_csv(primary_interactions, "10_tier1_R_primary_interaction_multiple_testing.csv")
save_csv(primary_joint, "11_tier1_R_cvd_primary_joint_hr_table.csv")
save_csv(complete_case_counts, "12_tier1_R_complete_case_counts.csv")
save_csv(interpretation_flags, "13_tier1_R_interpretation_flags.csv")
save_csv(figure_df, "20_tier1_R_figure_manifest.csv")

manifest_files <- list.files(TIER1_DIR, pattern = "[.](csv|png|pdf)$", full.names = TRUE)
manifest <- data.frame(
  output_file = basename(manifest_files),
  path = manifest_files,
  size_bytes = file.info(manifest_files)$size,
  stringsAsFactors = FALSE
)
save_csv(manifest, "99_tier1_R_model_manifest.csv")

cat("DONE\n")
cat("Tier 1 R model outputs are in:", TIER1_DIR, "\n")
cat("Upload these first: 04_tier1_R_model_fit_status.csv, 07_tier1_R_cvd_cox_primary_term_results.csv, 10_tier1_R_primary_interaction_multiple_testing.csv, 11_tier1_R_cvd_primary_joint_hr_table.csv, 12_tier1_R_complete_case_counts.csv, 14_tier1_R_cox_ph_assumption_zph.csv, 20_tier1_R_figure_manifest.csv, 21_tier1_R_primary_interaction_forest.png, 22_tier1_R_primary_joint_hr_forest.png, 23_tier1_R_cox_ph_global_pvalues.png, 24_tier1_R_cox_zph_<rsid>_cvd_binary_core_with_bmi.pdf, 99_tier1_R_model_manifest.csv\n")
primary_interactions


## PH sensitivity models


In [ ]:
TIER1_SNPS <- c("rs1544410", "rs2228570", "rs731236", "rs7975232")
MODEL_PREP_OUTPUT_DIR_NAME <- "Q3_VDR_v9_model_prep_outputs"
PH_OUTPUT_DIR_NAME <- "Q3_VDR_tier1_ph_sensitivity_outputs_R"

OVERWRITE_OUTPUTS <- tolower(Sys.getenv("Q3_VDR_OVERWRITE_OUTPUTS", "true")) %in% c("1", "true", "yes", "y")

infer_default_data_dir <- function() {
  cwd <- getwd()
  candidates <- c(
    cwd,
    file.path(cwd, "data"),
    "/home/jupyter/workspace/VDR",
    "/home/jupyter/workspace/VDR/data",
    "/home/jupyter/workspace/Q3_VDR",
    "/home/jupyter/workspace/Q3_VDR/data"
  )
  for (p in candidates) {
    if (dir.exists(p)) return(p)
  }
  cwd
}

DATA_DIR <- Sys.getenv("Q3_VDR_DATA_DIR", infer_default_data_dir())
MODEL_PREP_DIR <- Sys.getenv("Q3_VDR_MODEL_OUTPUT_DIR", file.path(DATA_DIR, MODEL_PREP_OUTPUT_DIR_NAME))
PH_DIR <- Sys.getenv("Q3_VDR_PH_SENSITIVITY_OUTPUT_DIR", file.path(DATA_DIR, PH_OUTPUT_DIR_NAME))
dir.create(PH_DIR, recursive = TRUE, showWarnings = FALSE)

save_csv <- function(df, filename) {
  path <- file.path(PH_DIR, filename)
  existed <- file.exists(path)
  if (existed && !OVERWRITE_OUTPUTS) stop(sprintf("Output exists and overwrite is disabled: %s", path))
  write.csv(df, path, row.names = FALSE, na = "")
  action <- if (existed) "overwritten" else "created"
  message(sprintf("%s: %s rows=%s cols=%s", action, path, nrow(df), ncol(df)))
  path
}

save_text <- function(lines, filename) {
  path <- file.path(PH_DIR, filename)
  existed <- file.exists(path)
  if (existed && !OVERWRITE_OUTPUTS) stop(sprintf("Output exists and overwrite is disabled: %s", path))
  writeLines(lines, con = path, useBytes = TRUE)
  action <- if (existed) "overwritten" else "created"
  message(sprintf("%s: %s", action, path))
  path
}

pkg_status <- data.frame(
  package = c("survival", "stats", "grDevices", "graphics"),
  available = c(requireNamespace("survival", quietly = TRUE), TRUE, TRUE, TRUE),
  stringsAsFactors = FALSE
)
if (!pkg_status$available[pkg_status$package == "survival"]) {
  stop("R package 'survival' is required for this notebook but is not available.")
}
library(survival)

run_config <- data.frame(
  setting = c(
    "DATA_DIR", "MODEL_PREP_DIR", "PH_DIR", "OVERWRITE_OUTPUTS",
    "TIER1_SNPS", "scope", "model_families", "heavy_tvc_policy", "output_policy"
  ),
  value = c(
    DATA_DIR, MODEL_PREP_DIR, PH_DIR, as.character(OVERWRITE_OUTPUTS),
    paste(TIER1_SNPS, collapse = ";"),
    "Tier 1 VDR SNP x measured 25OHD CVD PH-robust sensitivities only; no MR/GRS/pathway expansion.",
    "primary Cox refit; age-stratified Cox; 2y/5y truncated Cox; 0-2/2-5/>5y window-specific Cox; Poisson rate with offset",
    "coxph(tt=...) TVC model skipped by default after Workbench kernel restart; window-specific Cox is the feasible time-varying sensitivity.",
    "overwrite same-name files in PH_DIR; do not delete unrelated files"
  ),
  stringsAsFactors = FALSE
)

save_csv(run_config, "00_ph_sensitivity_run_configuration.csv")
save_csv(pkg_status, "00_ph_sensitivity_package_status.csv")
cat("Reading model-prep inputs from:", MODEL_PREP_DIR, "\n")
cat("Writing PH sensitivity outputs to:", PH_DIR, "\n")
read_csv_required <- function(path, label) {
  if (!file.exists(path)) stop(sprintf("Missing required input for %s: %s", label, path))
  df <- read.csv(path, check.names = FALSE, stringsAsFactors = FALSE)
  message(sprintf("Loaded %s: rows=%s cols=%s from %s", label, nrow(df), ncol(df), path))
  df
}

cvd_path <- file.path(MODEL_PREP_DIR, "02_cvd_model_dataset_v9.csv")
cvd <- read_csv_required(cvd_path, "CVD model-prep dataset")

tier1_carrier_cols <- paste0(TIER1_SNPS, "_carrier")
missing_tier1 <- setdiff(tier1_carrier_cols, names(cvd))
if (length(missing_tier1) > 0) {
  stop(sprintf("Missing Tier 1 carrier columns in CVD model dataset: %s", paste(missing_tier1, collapse = ";")))
}

numeric_cols <- c(
  "vitd_main", "log_vitd", "vitd_lt20", "vitd_per10", "age_at_index", "female", "winter", "bmi_full",
  "follow_up_years", "event_mi_or_stroke", "baseline_diabetes", "baseline_hypertension",
  "baseline_hyperlipidemia", tier1_carrier_cols
)
for (col in intersect(numeric_cols, names(cvd))) {
  cvd[[col]] <- suppressWarnings(as.numeric(cvd[[col]]))
}

for (rsid in TIER1_SNPS) {
  carrier_col <- paste0(rsid, "_carrier")
  ix_bin <- paste0(rsid, "_x_vitd_lt20")
  cvd[[carrier_col]] <- as.integer(cvd[[carrier_col]])
  cvd[[ix_bin]] <- cvd$vitd_lt20 * cvd[[carrier_col]]
}

cvd$age_band <- cut(
  cvd$age_at_index,
  breaks = c(-Inf, 50, 65, 75, Inf),
  labels = c("<50", "50-64", "65-74", "75+"),
  right = FALSE
)

primary_cc_cols <- c(
  "follow_up_years", "event_mi_or_stroke", "vitd_lt20", "age_at_index",
  "female", "winter", "bmi_full", tier1_carrier_cols[1],
  paste0(TIER1_SNPS[1], "_x_vitd_lt20")
)
primary_cc_index <- complete.cases(cvd[, primary_cc_cols, drop = FALSE]) &
  !is.na(cvd$follow_up_years) & cvd$follow_up_years > 0

input_summary <- data.frame(
  metric = c(
    "cvd_rows",
    "events",
    "follow_up_positive_n",
    "primary_complete_case_n",
    "primary_complete_case_events",
    "age_band_nonmissing_n"
  ),
  value = c(
    nrow(cvd),
    sum(cvd$event_mi_or_stroke, na.rm = TRUE),
    sum(!is.na(cvd$follow_up_years) & cvd$follow_up_years > 0),
    sum(primary_cc_index, na.rm = TRUE),
    sum(cvd$event_mi_or_stroke[primary_cc_index], na.rm = TRUE),
    sum(!is.na(cvd$age_band))
  ),
  stringsAsFactors = FALSE
)
save_csv(input_summary, "01_ph_sensitivity_input_summary.csv")

age_band_summary <- as.data.frame(table(cvd$age_band, useNA = "ifany"), stringsAsFactors = FALSE)
names(age_band_summary) <- c("age_band", "n")
save_csv(age_band_summary, "01b_ph_sensitivity_age_band_summary.csv")

cat("Loaded CVD rows:", nrow(cvd), "\n")
input_summary
bind_rows_base <- function(rows) {
  rows <- rows[vapply(rows, function(x) is.data.frame(x) && nrow(x) > 0, logical(1))]
  if (length(rows) == 0) return(data.frame())
  all_cols <- unique(unlist(lapply(rows, names)))
  aligned <- lapply(rows, function(d) {
    missing_cols <- setdiff(all_cols, names(d))
    for (col in missing_cols) d[[col]] <- NA
    d[, all_cols, drop = FALSE]
  })
  do.call(rbind, aligned)
}

complete_case_data <- function(df, needed) {
  missing_terms <- setdiff(needed, names(df))
  if (length(missing_terms) > 0) {
    return(list(data = data.frame(), status = "missing_terms", detail = paste(missing_terms, collapse = ";")))
  }
  d <- df[stats::complete.cases(df[, needed, drop = FALSE]), , drop = FALSE]
  d <- d[!is.na(d$follow_up_years) & d$follow_up_years > 0, , drop = FALSE]
  if (nrow(d) == 0) return(list(data = d, status = "no_complete_cases", detail = ""))
  list(data = d, status = "ready", detail = "")
}

inv_xtx <- function(x) {
  tryCatch(solve(x), error = function(e) qr.solve(x))
}

effect_from_vcov <- function(label, coef_vec, vc, weights, scale_label = "HR") {
  terms <- names(weights)
  if (length(terms) == 0 || any(!(terms %in% names(coef_vec))) || any(!(terms %in% rownames(vc)))) {
    return(data.frame(
      estimand = label, estimate_scale = scale_label, coef = NA_real_, se = NA_real_,
      p = NA_real_, estimate = NA_real_, ci_lower = NA_real_, ci_upper = NA_real_,
      status = "missing_terms", detail = paste(setdiff(terms, names(coef_vec)), collapse = ";"),
      stringsAsFactors = FALSE
    ))
  }
  w <- as.numeric(weights)
  names(w) <- terms
  beta <- sum(w * coef_vec[terms])
  sub_v <- vc[terms, terms, drop = FALSE]
  variance <- as.numeric(t(w) %*% sub_v %*% w)
  se <- ifelse(is.finite(variance) && variance >= 0, sqrt(variance), NA_real_)
  z <- beta / se
  p <- 2 * pnorm(abs(z), lower.tail = FALSE)
  data.frame(
    estimand = label,
    estimate_scale = scale_label,
    coef = beta,
    se = se,
    p = p,
    estimate = exp(beta),
    ci_lower = exp(beta - 1.96 * se),
    ci_upper = exp(beta + 1.96 * se),
    status = "pass",
    detail = "",
    stringsAsFactors = FALSE
  )
}

cox_terms_table <- function(fit) {
  s <- summary(fit)
  coefs <- as.data.frame(s$coefficients)
  coefs$term <- rownames(coefs)
  rownames(coefs) <- NULL
  data.frame(
    term = coefs$term,
    coef = coefs$coef,
    se = coefs$`se(coef)`,
    z = coefs$z,
    p = coefs$`Pr(>|z|)`,
    hr = exp(coefs$coef),
    hr_ci_lower = exp(coefs$coef - 1.96 * coefs$`se(coef)`),
    hr_ci_upper = exp(coefs$coef + 1.96 * coefs$`se(coef)`),
    stringsAsFactors = FALSE
  )
}

cox_interaction_effect <- function(fit, interaction_term, label = "interaction") {
  effect_from_vcov(label, coef(fit), vcov(fit), setNames(1, interaction_term), "HR_or_HR_ratio")
}

fit_cox_model <- function(df, rsid, model_family, mode = c("primary", "age_stratified", "truncated", "window"), tau = NA, window_start = NA, window_end = NA, window_label = NA) {
  mode <- match.arg(mode)
  carrier <- paste0(rsid, "_carrier")
  interaction <- paste0(rsid, "_x_vitd_lt20")
  needed <- c("follow_up_years", "event_mi_or_stroke", "vitd_lt20", carrier, interaction, "age_at_index", "female", "winter", "bmi_full")
  if (mode == "age_stratified") needed <- c(needed, "age_band")
  cc <- complete_case_data(df, needed)
  info <- list(status = cc$status, detail = cc$detail, n = nrow(cc$data), events = NA, backend = "survival::coxph")
  if (cc$status != "ready") return(list(fit = NULL, info = info, effect = data.frame(), terms = data.frame()))
  d <- cc$data

  if (mode == "truncated") {
    d$analysis_time <- pmin(d$follow_up_years, tau)
    d$analysis_event <- as.integer(d$event_mi_or_stroke == 1 & d$follow_up_years <= tau)
    d <- d[d$analysis_time > 0, , drop = FALSE]
    info$backend <- paste0("survival::coxph_truncated_", tau, "y")
  } else if (mode == "window") {
    if (is.infinite(window_end)) {
      keep <- d$follow_up_years > window_start
      d <- d[keep, , drop = FALSE]
      d$analysis_time <- d$follow_up_years - window_start
      d$analysis_event <- as.integer(d$event_mi_or_stroke == 1 & d$follow_up_years > window_start)
    } else {
      keep <- d$follow_up_years > window_start
      d <- d[keep, , drop = FALSE]
      d$analysis_time <- pmin(d$follow_up_years, window_end) - window_start
      d$analysis_event <- as.integer(d$event_mi_or_stroke == 1 & d$follow_up_years > window_start & d$follow_up_years <= window_end)
    }
    d <- d[d$analysis_time > 0, , drop = FALSE]
    info$backend <- paste0("survival::coxph_window_", window_label)
  } else {
    d$analysis_time <- d$follow_up_years
    d$analysis_event <- d$event_mi_or_stroke
    if (mode == "age_stratified") info$backend <- "survival::coxph_age_band_strata"
  }

  info$n <- nrow(d)
  info$events <- sum(d$analysis_event, na.rm = TRUE)
  if (info$n < 100 || info$events < 50) {
    info$status <- "too_few_events"
    info$detail <- ""
    return(list(fit = NULL, info = info, effect = data.frame(), terms = data.frame()))
  }

  rhs <- paste0("vitd_lt20 + ", carrier, " + ", interaction, " + age_at_index + female + winter + bmi_full")
  if (mode == "age_stratified") rhs <- paste0("vitd_lt20 + ", carrier, " + ", interaction, " + female + winter + bmi_full + strata(age_band)")
  f <- as.formula(paste0("Surv(analysis_time, analysis_event) ~ ", rhs))
  fit <- tryCatch(coxph(f, data = d, ties = "efron", x = FALSE, model = FALSE), error = function(e) e)
  if (inherits(fit, "error")) {
    info$status <- "fit_error"
    info$detail <- conditionMessage(fit)
    return(list(fit = NULL, info = info, effect = data.frame(), terms = data.frame()))
  }
  info$status <- "pass"
  eff <- cox_interaction_effect(fit, interaction, paste0(model_family, "_interaction_ratio"))
  terms <- cox_terms_table(fit)
  list(fit = fit, info = info, effect = eff, terms = terms)
}

glm_poisson_vcov_hc0 <- function(fit) {
  X <- model.matrix(fit)
  y <- model.response(model.frame(fit))
  mu <- fitted(fit)
  bread <- inv_xtx(crossprod(X, X * as.numeric(mu)))
  score <- X * as.numeric(y - mu)
  meat <- crossprod(score)
  bread %*% meat %*% bread
}

fit_poisson_rate <- function(df, rsid) {
  carrier <- paste0(rsid, "_carrier")
  interaction <- paste0(rsid, "_x_vitd_lt20")
  needed <- c("follow_up_years", "event_mi_or_stroke", "vitd_lt20", carrier, interaction, "age_at_index", "female", "winter", "bmi_full")
  cc <- complete_case_data(df, needed)
  info <- list(status = cc$status, detail = cc$detail, n = nrow(cc$data), events = NA, backend = "glm_poisson_offset_sandwich_hc0")
  if (cc$status != "ready") return(list(fit = NULL, info = info, effect = data.frame()))
  d <- cc$data
  info$events <- sum(d$event_mi_or_stroke, na.rm = TRUE)
  f <- as.formula(paste0("event_mi_or_stroke ~ vitd_lt20 + ", carrier, " + ", interaction, " + age_at_index + female + winter + bmi_full + offset(log(follow_up_years))"))
  fit <- tryCatch(glm(f, data = d, family = poisson(link = "log")), error = function(e) e)
  if (inherits(fit, "error")) {
    info$status <- "fit_error"
    info$detail <- conditionMessage(fit)
    return(list(fit = NULL, info = info, effect = data.frame()))
  }
  vc <- tryCatch(glm_poisson_vcov_hc0(fit), error = function(e) e)
  if (inherits(vc, "error")) {
    info$status <- "sandwich_vcov_error"
    info$detail <- conditionMessage(vc)
    return(list(fit = fit, info = info, effect = data.frame()))
  }
  eff <- tryCatch(
    effect_from_vcov("poisson_rate_offset_interaction_ratio", coef(fit), vc, setNames(1, interaction), "RR_or_RR_ratio"),
    error = function(e) e
  )
  if (inherits(eff, "error")) {
    info$status <- "effect_error"
    info$detail <- conditionMessage(eff)
    return(list(fit = fit, info = info, effect = data.frame()))
  }
  info$status <- "pass"
  list(fit = fit, info = info, effect = eff)
}

add_status <- function(model_family, rsid, info, horizon = NA, time_window = NA) {
  data.frame(
    model_family = model_family,
    rsid = rsid,
    status = info$status,
    detail = info$detail,
    n = info$n,
    events = info$events,
    backend = info$backend,
    horizon_years = horizon,
    time_window = time_window,
    stringsAsFactors = FALSE
  )
}

decorate_effect <- function(df, model_family, rsid, info, horizon = NA, time_window = NA) {
  if (nrow(df) == 0) return(df)
  df$model_family <- model_family
  df$rsid <- rsid
  df$n <- info$n
  df$events <- info$events
  df$backend <- info$backend
  df$horizon_years <- horizon
  df$time_window <- time_window
  df
}
spec_rows <- data.frame(
  model_family = c(
    "primary_cox_refit",
    "age_band_stratified_cox",
    "truncated_horizon_cox",
    "piecewise_window_cox",
    "poisson_rate_offset",
    "time_varying_coefficient_cox_not_run"
  ),
  role = c(
    "primary model preserved for comparison",
    "PH sensitivity for age_at_index signal",
    "2y and 5y horizon-specific Cox estimates",
    "0-2, 2-5, and >5 year follow-up window estimates",
    "non-PH rate-ratio sensitivity using follow-up offset",
    "methodologically direct but skipped by default after kernel restart from coxph(tt=...) memory expansion"
  ),
  primary_interpretation = c("no", "sensitivity", "sensitivity", "sensitivity", "sensitivity", "not_run"),
  stringsAsFactors = FALSE
)
save_csv(spec_rows, "02_ph_sensitivity_model_specifications.csv")

all_status <- list()
comparison_rows <- list()
term_rows <- list()

for (rsid in TIER1_SNPS) {
  primary <- fit_cox_model(cvd, rsid, "primary_cox_refit", mode = "primary")
  all_status[[length(all_status) + 1]] <- add_status("primary_cox_refit", rsid, primary$info)
  comparison_rows[[length(comparison_rows) + 1]] <- decorate_effect(primary$effect, "primary_cox_refit", rsid, primary$info)
  if (nrow(primary$terms) > 0) {
    terms <- primary$terms
    terms$model_family <- "primary_cox_refit"
    terms$rsid <- rsid
    term_rows[[length(term_rows) + 1]] <- terms
  }

  age_strat <- fit_cox_model(cvd, rsid, "age_band_stratified_cox", mode = "age_stratified")
  all_status[[length(all_status) + 1]] <- add_status("age_band_stratified_cox", rsid, age_strat$info)
  comparison_rows[[length(comparison_rows) + 1]] <- decorate_effect(age_strat$effect, "age_band_stratified_cox", rsid, age_strat$info)
}

status_df <- bind_rows_base(all_status)
comparison_df <- bind_rows_base(comparison_rows)
terms_df <- bind_rows_base(term_rows)
save_csv(status_df, "03_ph_sensitivity_fit_status.csv")
save_csv(comparison_df, "04_ph_sensitivity_primary_age_comparison_partial.csv")
save_csv(terms_df, "07c_ph_sensitivity_selected_model_terms.csv")
head(status_df, 20)
block4_status <- list()
block4_rows <- list()

for (rsid in TIER1_SNPS) {
  for (tau in c(2, 5)) {
    trunc_fit <- fit_cox_model(cvd, rsid, paste0("truncated_horizon_cox_", tau, "y"), mode = "truncated", tau = tau)
    fam <- "truncated_horizon_cox"
    block4_status[[length(block4_status) + 1]] <- add_status(fam, rsid, trunc_fit$info, horizon = tau)
    block4_rows[[length(block4_rows) + 1]] <- decorate_effect(trunc_fit$effect, fam, rsid, trunc_fit$info, horizon = tau)
  }

  window_specs <- data.frame(
    window_start = c(0, 2, 5),
    window_end = c(2, 5, Inf),
    window_label = c("0-2y", "2-5y", ">5y"),
    stringsAsFactors = FALSE
  )
  for (k in seq_len(nrow(window_specs))) {
    wfit <- fit_cox_model(
      cvd, rsid, paste0("piecewise_window_cox_", window_specs$window_label[k]),
      mode = "window",
      window_start = window_specs$window_start[k],
      window_end = window_specs$window_end[k],
      window_label = window_specs$window_label[k]
    )
    fam <- "piecewise_window_cox"
    block4_status[[length(block4_status) + 1]] <- add_status(fam, rsid, wfit$info, time_window = window_specs$window_label[k])
    block4_rows[[length(block4_rows) + 1]] <- decorate_effect(wfit$effect, fam, rsid, wfit$info, time_window = window_specs$window_label[k])
  }

  pois <- fit_poisson_rate(cvd, rsid)
  block4_status[[length(block4_status) + 1]] <- add_status("poisson_rate_offset", rsid, pois$info)
  block4_rows[[length(block4_rows) + 1]] <- decorate_effect(pois$effect, "poisson_rate_offset", rsid, pois$info)
}

block4_status_df <- bind_rows_base(block4_status)
block4_comparison_df <- bind_rows_base(block4_rows)

status_df <- bind_rows_base(list(status_df, block4_status_df))
comparison_df <- bind_rows_base(list(comparison_df, block4_comparison_df))

tvc_skipped_status <- data.frame(
  model_family = "time_varying_coefficient_cox_not_run",
  rsid = TIER1_SNPS,
  status = "skipped_default_memory_guardrail",
  detail = "coxph(tt=...) caused Workbench kernel restart in prior version; use window-specific Cox as feasible time-varying sensitivity",
  n = NA,
  events = NA,
  backend = "not_run",
  horizon_years = NA,
  time_window = NA,
  stringsAsFactors = FALSE
)
status_df <- bind_rows_base(list(status_df, tvc_skipped_status))

save_csv(status_df, "03_ph_sensitivity_fit_status.csv")
save_csv(comparison_df, "08_ph_sensitivity_compact_comparison_table.csv")
head(comparison_df, 40)
comparison <- comparison_df
if (nrow(comparison) > 0) {
  comparison$model_role <- ifelse(comparison$model_family == "primary_cox_refit", "primary_refit", "ph_robust_sensitivity")
  comparison$ci_crosses_null <- with(comparison, ifelse(is.finite(ci_lower) & is.finite(ci_upper), ci_lower <= 1 & ci_upper >= 1, NA))
  comparison$nominal_p_lt_0_05 <- ifelse(is.finite(comparison$p), comparison$p < 0.05, NA)
  comparison$display_estimate <- sprintf("%.3f (%.3f, %.3f)", comparison$estimate, comparison$ci_lower, comparison$ci_upper)
  comparison$display_p <- ifelse(is.finite(comparison$p), sprintf("%.3g", comparison$p), "")
  comparison <- comparison[order(comparison$rsid, comparison$model_family, comparison$horizon_years, comparison$time_window), , drop = FALSE]
}
save_csv(comparison, "08_ph_sensitivity_compact_comparison_table.csv")

interpretation_flags <- data.frame(
  item = c(
    "primary_model_preserved",
    "heavy_tvc_model",
    "ph_violation_response",
    "main_estimand",
    "genetic_coding",
    "mr_causal_language",
    "comparison_rule"
  ),
  status = c(
    "yes",
    "skipped_default_memory_guardrail",
    "memory_safe_sensitivity_models_added",
    "effect_modification",
    "dominant_ALT_carrier",
    "prohibited",
    "compare_direction_and_ci_not_single_p_value"
  ),
  note = c(
    "Original Cox PH model remains primary and is refit only for side-by-side comparison.",
    "coxph(tt=...) caused a Workbench kernel restart in the prior version; it is not run by default.",
    "Age-stratified Cox, truncated-horizon Cox, window-specific Cox, and Poisson rate-offset sensitivity models are added.",
    "The target remains VDR carrier-status modification of the measured 25OHD-CVD association.",
    "Carrier status is not additive genotype dosage or externally weighted genetic instrument.",
    "Do not describe these analyses as MR or causal proof.",
    "Robustness is judged by whether interaction estimates remain near null across PH-robust alternatives."
  ),
  stringsAsFactors = FALSE
)
save_csv(interpretation_flags, "09_ph_sensitivity_interpretation_flags.csv")

make_figure_row <- function(filename, figure_type, status, detail = "", n_rows = NA) {
  data.frame(
    output_file = filename,
    path = file.path(PH_DIR, filename),
    figure_type = figure_type,
    status = status,
    detail = detail,
    n_rows = n_rows,
    stringsAsFactors = FALSE
  )
}

plot_comparison_forest <- function(df) {
  filename <- "21_ph_sensitivity_compact_interaction_forest.png"
  if (nrow(df) == 0) return(make_figure_row(filename, "ph_sensitivity_interaction_forest", "no_data", "", 0))
  d <- df[is.finite(df$estimate) & is.finite(df$ci_lower) & is.finite(df$ci_upper) & df$ci_lower > 0 & df$ci_upper > 0, , drop = FALSE]
  if (nrow(d) == 0) return(make_figure_row(filename, "ph_sensitivity_interaction_forest", "no_plottable_rows", "", 0))
  d$label <- ifelse(
    d$model_family == "truncated_horizon_cox",
    paste0(d$rsid, " | Cox truncated ", d$horizon_years, "y"),
    ifelse(
      d$model_family == "piecewise_window_cox",
      paste0(d$rsid, " | Cox window ", d$time_window),
      ifelse(
        d$model_family == "poisson_rate_offset",
        paste0(d$rsid, " | Poisson rate"),
        paste0(d$rsid, " | ", d$model_family)
      )
    )
  )
  d <- d[order(d$rsid, d$model_family, d$horizon_years, d$time_window), , drop = FALSE]
  path <- file.path(PH_DIR, filename)
  status <- "created"
  detail <- ""
  tryCatch({
    grDevices::png(path, width = 2300, height = max(1200, 240 + 68 * nrow(d)), res = 170)
    on.exit(grDevices::dev.off(), add = TRUE)
    par(mar = c(5, 13, 4, 2))
    y <- seq_len(nrow(d))
    log_range <- range(log(c(d$ci_lower, d$ci_upper, 1)), finite = TRUE)
    pad <- max(0.12, diff(log_range) * 0.08)
    xlim <- exp(log_range + c(-pad, pad))
    plot(d$estimate, y, log = "x", xlim = xlim, yaxt = "n", pch = 19,
         xlab = "Interaction estimate ratio, log scale", ylab = "",
         main = "Tier 1 VDR x vitamin D deficiency: PH-robust sensitivity comparison")
    segments(d$ci_lower, y, d$ci_upper, y, lwd = 1.2)
    abline(v = 1, lty = 2, col = "gray45")
    axis(2, at = y, labels = d$label, las = 1, cex.axis = 0.58)
  }, error = function(e) {
    status <<- "plot_error"
    detail <<- conditionMessage(e)
  })
  make_figure_row(filename, "ph_sensitivity_interaction_forest", status, detail, nrow(d))
}

fig_manifest <- plot_comparison_forest(comparison)
save_csv(fig_manifest, "20_ph_sensitivity_figure_manifest.csv")

primary_subset <- comparison[comparison$model_family == "primary_cox_refit", , drop = FALSE]
sensitivity_subset <- comparison[comparison$model_family != "primary_cox_refit", , drop = FALSE]
all_primary_cross <- if (nrow(primary_subset) > 0) all(primary_subset$ci_crosses_null %in% TRUE) else NA
all_sensitivity_cross <- if (nrow(sensitivity_subset) > 0) all(sensitivity_subset$ci_crosses_null %in% TRUE) else NA
any_sensitivity_nominal <- if (nrow(sensitivity_subset) > 0) any(sensitivity_subset$nominal_p_lt_0_05 %in% TRUE) else NA

note_lines <- c(
  "# Q3 VDR Tier 1 PH-Robust Sensitivity Interpretation",
  "",
  paste0("Date: ", format(Sys.Date(), "%Y-%m-%d")),
  "",
  "## Scope",
  "",
  "This note extends the Tier 1 VDR carrier-status x measured 25(OH)D CVD analysis after proportional-hazards diagnostics flagged the primary Cox models. The original Cox model remains the primary model. These analyses are sensitivity analyses, not MR, not an instrumental-variable analysis, and not causal proof.",
  "",
  "Tier 1 SNPs: rs1544410, rs2228570, rs731236, rs7975232.",
  "",
  "## Workbench Stability Change",
  "",
  "The previous coxph(tt=...) time-varying coefficient implementation caused a Workbench kernel restart before fitted-model outputs were written. This version skips that heavy model by default and uses memory-safe time-window alternatives.",
  "",
  "## Sensitivity Models Added",
  "",
  "1. Age-band stratified Cox model: addresses repeated PH signal for age_at_index by allowing separate baseline hazards across prespecified age bands.",
  "2. Truncated-horizon Cox models: estimates interaction ratios within 2-year and 5-year follow-up horizons.",
  "3. Piecewise/window-specific Cox models: estimates interaction ratios separately for 0-2, 2-5, and >5 years of follow-up.",
  "4. Poisson rate model with log follow-up offset and sandwich variance: provides a non-PH rate-ratio sensitivity.",
  "",
  "## Automated Summary Flags",
  "",
  paste0("- All primary Cox refit interaction CIs cross 1: ", all_primary_cross),
  paste0("- All PH-sensitivity interaction CIs cross 1: ", all_sensitivity_cross),
  paste0("- Any PH-sensitivity nominal interaction P < 0.05: ", any_sensitivity_nominal),
  "",
  "## Interpretation Template",
  "",
  "If the compact comparison table shows interaction estimates centered near 1 with confidence intervals crossing 1 across the PH-robust sensitivity families, the Tier 1 conclusion should remain: there is no robust evidence that classic VDR ALT-carrier status modifies the measured vitamin D deficiency-CVD association.",
  "",
  "If isolated PH-robust sensitivity rows show nominal P < 0.05, treat them as sensitivity signals requiring diagnostic review, not as a replacement for the primary null interaction result.",
  "",
  "## Methods References",
  "",
  "- Cox DR. Regression Models and Life-Tables. JRSS-B. 1972. DOI: https://doi.org/10.1111/j.2517-6161.1972.tb00899.x",
  "- Schoenfeld D. Partial residuals for the proportional hazards regression model. Biometrika. 1982. DOI: https://doi.org/10.1093/biomet/69.1.239",
  "- Grambsch PM, Therneau TM. Proportional hazards tests and diagnostics based on weighted residuals. Biometrika. 1994. DOI: https://doi.org/10.1093/biomet/81.3.515",
  "- Zou G. A modified Poisson regression approach to prospective studies with binary data. Am J Epidemiol. 2004. PMID: 15033648. PubMed: https://pubmed.ncbi.nlm.nih.gov/15033648/",
  "- Knol MJ, VanderWeele TJ. Recommendations for presenting analyses of effect modification and interaction. Int J Epidemiol. 2012. PMID: 22253321. PubMed: https://pubmed.ncbi.nlm.nih.gov/22253321/"
)
save_text(note_lines, "10_ph_sensitivity_updated_interpretation_note.md")

manifest_files <- list.files(PH_DIR, pattern = "[.](csv|png|pdf|md)$", full.names = TRUE)
manifest <- data.frame(
  output_file = basename(manifest_files),
  path = manifest_files,
  size_bytes = file.info(manifest_files)$size,
  stringsAsFactors = FALSE
)
save_csv(manifest, "99_ph_sensitivity_manifest.csv")

cat("DONE\n")
cat("PH sensitivity outputs are in:", PH_DIR, "\n")
cat("Upload these first: 03_ph_sensitivity_fit_status.csv, 08_ph_sensitivity_compact_comparison_table.csv, 10_ph_sensitivity_updated_interpretation_note.md, 20_ph_sensitivity_figure_manifest.csv, 21_ph_sensitivity_compact_interaction_forest.png, 99_ph_sensitivity_manifest.csv\n")
head(comparison, 40)
